In [ ]:
# Библиотеки для работы с таблицами, расчётов и графиков
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Единый стиль графиков и показ до 50 столбцов в таблицах
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns',50)

In [ ]:
# Данные: ежемесячные показатели жилищного строительства (ДОМ.РФ / наш.дом.рф,
# Файл лежит в GitHub-репозитории в виде ZIP.
URL = 'https://github.com/KOTGAME212/Building_Analysis_SPBSTU/raw/refs/heads/main/Building_SPBSTU.zip'

df = pd.read_csv(URL, compression='zip')
print(df.shape)
df.head()

In [ ]:
# Первичный осмотр: типы, пропуски, дубликаты, сводная статистика.
df.info()
print('Пропуски:\n', df.isna().sum())
print('Дубликаты:\n', df.duplicated().sum())
df.describe(include='all').T

In [ ]:
# Столбцы parent_object_* полностью пустые (все значения NaN) — удаляем.
# Дальше смотрим, какие категории есть в данных, чтобы корректно фильтровать:
# в одной таблице смешаны страна, федеральные округа и регионы, а механизм «Все механизмы»
# является суммой остальных трёх. Если не фильтровать, значения будут посчитаны по несколько раз.
df = df.drop(columns=["parent_object_name", "parent_object_code"])

print(df["object_level"].value_counts(), "\n")
print(df["funding_mechanism"].value_counts(), "\n")
print(df.groupby("indicator_name")["indicator_unit"].first(), "\n")
print(df.groupby("object_level")["object_name"].unique())

print(df[["year", "month"]].drop_duplicates().sort_values(["year", "month"]).tail(3))

In [ ]:
# Дата первого числа месяца из year и month — удобно для временных рядов
# Берём только регионы и только «Все механизмы», чтобы не было двойного счёта
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))

reg = df[(df["object_level"] == "регион") &
         (df["funding_mechanism"] == "Все механизмы")]

wide = reg.pivot_table(index=["object_name", "object_code", "date"],
                       columns="indicator_name",
                       values="indicator_value",
                       aggfunc="first").reset_index()

print(wide.shape)
print(wide.columns.tolist())
wide.head()

In [ ]:
# Короткие имена столбцов для удобства
rename = {
    "Количество застройщиков": "developers",
    "Количество разрешений на строительство": "permits",
    "Количество проектных деклараций": "declarations",
    "Количество объектов строительства": "objects",
    "Общая площадь строительства": "area_total",
    "Жилая площадь строительства": "area_living",
    "Количество жилых помещений": "flats",
}
wide = wide.rename(columns=rename).rename(columns={"object_name": "region"})
wide.describe().T

In [ ]:
# Динамика по стране: берём уровень «страна» напрямую из исходной таблицы
# Показатель — моментальный срез за месяц

country = df[(df["object_level"] == "страна") &
             (df["funding_mechanism"] == "Все механизмы") &
             (df["indicator_name"] == "Общая площадь строительства")]
country.set_index("date")["indicator_value"].plot(figsize=(10, 4),
    title="Общая площадь строительства в РФ, тыс. м²")
plt.show()

# Топ-10 регионов по площади строительства в последнем месяце данных

last = wide[wide["date"] == wide["date"].max()].nlargest(10, "area_total")
sns.barplot(data=last, x="area_total", y="region")
plt.title("Топ-10 регионов по площади строительства")
plt.show()


# Корреляции между показателями, отраженная на тепловой карте

cols = ["developers", "permits", "declarations", "objects", "area_total", "flats"]
sns.heatmap(np.log1p(wide[cols]).corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title('Корреляции показателей жилищного строительства в регионах РФ')
plt.show()